# CICE aice vs POP IFRAC — ck70

Sea-ice fraction from two places in the same run, both on the native gx1v7 grid (no regridding):

* **CICE `aice`** — the ice model's own concentration (`*.cice.h.YYYY-MM.nc`)
* **POP `IFRAC`** — the ice fraction the coupler passed to the ocean (`*.pop.h.YYYY-MM.nc`)

Step 1 builds a per-year cache on scratch (`icefrac.CACHE_ROOT`) — run it once; it skips years already cached.
Everything after reads the cache. Code lives in `ice_ens/icefrac.py`. Kernel: **npl-2026a**.

In [ ]:
import matplotlib.pyplot as plt
from ice_ens import icefrac

CASE  = "ck70"
YEARS = (1980, 2015)
HEMI  = "north"          # "north" or "south"
FIG_DIR = f"/glade/work/tvoeller/honors_thesis/code/output/{CASE}/ice_plot/aice_vs_ifrac/" + \
          ("arctic" if HEMI == "north" else "antarctic")

## 1. Build the cache (slow the first time — reads the 1.6 GB POP files once)

In [ ]:
icefrac.build_icefrac_cache(CASE, YEARS)          # force=True to rebuild

## 2. Load

In [ ]:
ds = icefrac.load_icefrac(CASE, YEARS)
ds

## 3. Side-by-side map: aice | IFRAC | aice − IFRAC
Set `MONTHS` — `[9]` for September, `[6, 7, 8]` for a JJA mean.

In [ ]:
MONTHS = [9]
fig = icefrac.plot_icefrac_compare(ds, MONTHS, HEMI, fig_dir=FIG_DIR)
plt.show()

## 4. 12-month climatology of one field
`VAR` = `"aice"`, `"IFRAC"` or `"diff"`.

In [ ]:
VAR = "aice"
fig = icefrac.plot_aice_months(ds, HEMI, var=VAR, fig_dir=FIG_DIR)
plt.show()

## 5. Hemispheric ice area through time
`months=None` plots every month; `months=[3]` or `[9]` gives one point per year.

In [ ]:
fig = icefrac.plot_area_series(ds, HEMI, months=None, fig_dir=FIG_DIR)
plt.show()
fig = icefrac.plot_area_series(ds, HEMI, months=[9], fig_dir=FIG_DIR)
plt.show()

In [ ]:
# The numbers behind the area plot
icefrac.ice_area_series(ds, HEMI).to_dataframe().round(3).head(24)